<a href="https://colab.research.google.com/github/amit-sw/colab_notebooks/blob/main/20260308_Chatbot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Set up

In [9]:
!pip install langchain_openai langgraph pydantic langchain_core langchain_community --quiet

In [10]:
from IPython.display import display, Markdown
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START, END
from pydantic import BaseModel

from langchain_core.messages import SystemMessage, BaseMessage, HumanMessage, AIMessage

In [11]:
import os
from google.colab import userdata

LANGCHAIN_API_KEY=userdata.get('LANGCHAIN_API_KEY')
OPENAI_API_KEY=userdata.get('OPENAI_API_KEY')


os.environ["LANGCHAIN_TRACING_V2"]="true"
os.environ["LANGCHAIN_API_KEY"]=LANGCHAIN_API_KEY
os.environ['OPENAI_API_KEY']=OPENAI_API_KEY
os.environ["LANGCHAIN_PROJECT"]="AIClub_Pro"
os.environ['LANGCHAIN_ENDPOINT']="https://api.smith.langchain.com"

# Single message : Request-response

In [5]:
# Single message
system_prompt="You are a helpful assistant. Please answer the user's questions in 2-3 paragraphs max."
user_input="what is the meaning of life?."

In [6]:
msg_list=[SystemMessage(content=system_prompt),
          HumanMessage(content=user_input)]
model=ChatOpenAI(model_name="gpt-5-nano", openai_api_key=OPENAI_API_KEY)
resp = model.invoke(msg_list)
display(Markdown(resp.content))


There isn’t a single universal answer to the meaning of life. Different traditions and thinkers offer different takes: some say meaning comes from a higher power or a cosmic purpose; others argue that meaning is something we create through our choices, relationships, and the kind of person we strive to be. Science often focuses on how we live rather than why the universe exists, but it doesn’t preclude individuals from finding personal significance in their experiences.

If you’re seeking meaning, you might explore what truly matters to you and how you want to spend your energy. Ways people find it include cultivating close relationships, contributing to something larger than oneself, pursuing growth and curiosity, and aligning daily actions with personal values. Small, consistent acts—listening deeply, helping others, learning something new—can add up to a lasting sense of purpose. If you want, tell me about your beliefs or what you care most about, and I can tailor some ideas.

# Single message: Streaming response

In [7]:
msg_list=[SystemMessage(content=system_prompt),
          HumanMessage(content=user_input)]
model=ChatOpenAI(model_name="gpt-5-nano", openai_api_key=OPENAI_API_KEY)
resp = model.stream(msg_list)

full_text = ""
handle = display(Markdown(""), display_id=True)

for chunk in resp:
    if chunk.content:
        full_text += chunk.content
        handle.update(Markdown(full_text))

There isn’t a single universal answer to the meaning of life. Different traditions and thinkers offer different takes: some say meaning comes from a higher purpose or God, others from relationships and service, and many secular view it as something we create for ourselves. Scientifically, life doesn’t come with an inherent purpose, but that doesn’t mean we can’t and shouldn’t make our own sense of it.

If you’re trying to find meaning, a practical approach is to identify what matters most to you—your values, the people you care about, and what you feel helps you grow. You can then align your daily choices with those priorities: nurture close relationships, contribute to something larger than yourself, learn and create, or help others. It’s a personal journey that can change over time; what matters is that your actions reflect what you value. What would you want your life to be about?

# Chat with memory: Streaming response

In [8]:
msg_list=[SystemMessage(content=system_prompt)]
model=ChatOpenAI(model_name="gpt-5-nano", openai_api_key=OPENAI_API_KEY)
while True:
  user_input=input("User: ")
  if(len(user_input)<=0):
    print("\n\n\nGoodbye!!\n\n\n")
    break
  msg_list.append(HumanMessage(content=user_input))
  resp = model.stream(msg_list)

  full_text = ""
  handle = display(Markdown(""), display_id=True)

  for chunk in resp:
      if chunk.content:
          full_text += chunk.content
          handle.update(Markdown(full_text))
  print("\n**********\n")

User: Hello


Hello! Nice to meet you. How can I help today? I can explain concepts, answer questions, help with writing or coding, brainstorm ideas, plan a trip, or just chat.


**********

User: 



Goodbye!!





# Graph

In [12]:
def create_llm_msg(system_prompt,history):
    resp=[SystemMessage(content=system_prompt)]
    msgs = history
    for m in msgs:
        if m["role"] == "user":
            resp.append(HumanMessage(content=m["content"]))
        elif m["role"] == "assistant":
            resp.append(AIMessage(content=m["content"]))
    #print(f"DEBUG CREATE LLM MSGS: {history=}\n{resp=}")
    return resp

In [13]:
class AgentState(BaseModel):
    """State of the agent."""
    messages: list = []
    response: str = ""
    category: str = ""

class Category(BaseModel):
    """Category for the agent."""
    category: str

In [14]:
class ChatbotAgent():
    """A chatbot agent that interacts with users."""

    def __init__(self, api_key: str):
        self.model = ChatOpenAI(model_name="gpt-5-nano", openai_api_key=api_key)
        workflow = StateGraph(AgentState)
        workflow.add_node("classifier", self.classifier)
        workflow.add_node("smalltalk_agent", self.smalltalk_agent)
        workflow.add_node("complaint_agent", self.complaint_agent)
        workflow.add_node("status_agent", self.status_agent)
        workflow.add_node("feedback_agent", self.feedback_agent)
        workflow.add_edge(START, "classifier")
        workflow.add_conditional_edges("classifier", self.main_router)
        #workflow.add_edge("classifier", "smalltalk_agent")
        #workflow.add_edge("classifier", "complaint_agent")
        #workflow.add_edge("classifier", "status_agent")
        #workflow.add_edge("classifier", "feedback_agent")
        workflow.add_edge("smalltalk_agent", END)
        workflow.add_edge("complaint_agent", END)
        workflow.add_edge("status_agent", END)
        workflow.add_edge("feedback_agent", END)

        self.graph = workflow.compile()



    def classifier(self, state: AgentState):
        #print("Initial classsifier")
        messages=state.messages
        CLASSIFIER_PROMPT = """
        You are a helpful assistant that classifies user messages into categories.
        Given the following messages, classify them into one of the following categories:
        - smalltalk_agent
        - complaint_agent
        - status_agent
        - feedback_agent

        If you don't know the category, classify it as "smalltalk_agent".
        """
        llm_messages = create_llm_msg(CLASSIFIER_PROMPT, state.messages)
        llm_response = self.model.with_structured_output(Category).invoke(llm_messages)
        category=llm_response.category
        print(f"Classified category: {category}")
        return {"category":category}

    def main_router(self, state: AgentState):
        #print("Routing to appropriate agent based on category")
        #print(f"DEBUG: Current state: {state}")
        #print(f"DEBUG: Current category: {state.category}")
        return state.category

    def smalltalk_agent(self, state: AgentState):
        print("Smalltalk agent processing....")
        SMALLTALK_PROMPT = f"""
        You are a friendly and engaging smalltalk agent designed to have casual conversations.
        Your purpose is to provide a warm and approachable interaction, ask follow-up questions to keep the conversation flowing,
        and inject a bit of humor or insight when appropriate. Avoid giving overly formal or short responses.
        If the user asks about something complex or outside the scope of small talk, gently redirect them to a more specific agent
        (e.g., 'That sounds like a question for our support team, but how's your day going otherwise?').
        Given the following messages, respond appropriately to the user's message, focusing on maintaining a pleasant chat.
        """
        llm_messages = create_llm_msg(SMALLTALK_PROMPT, state.messages)
        return {"response": self.model.stream(llm_messages), "category": "smalltalk_agent"}

    def complaint_agent(self, state: AgentState):
        print("Complaint agent processing....")
        COMPLAINT_PROMPT = f"""
        You are a dedicated complaint resolution agent. Your primary goal is to listen empathetically,
        acknowledge the user's frustration, and gather all necessary details to escalate or resolve their issue.
        Ensure you apologize for any inconvenience caused and reassure the user that their concern is being taken seriously.
        Ask clarifying questions to understand the full scope of the complaint.
        If possible, outline the next steps you will take to address their complaint.
        Given the following messages, respond appropriately to the user's complaint with a focus on resolution and empathy.
        """
        llm_messages = create_llm_msg(COMPLAINT_PROMPT, state.messages)
        return {"response": self.model.stream(llm_messages), "category": "complaint_agent"}

    def status_agent(self, state: AgentState):
        print("Status agent processing....")
        STATUS_PROMPT = f"""
        You are a precise and helpful status inquiry agent. Your role is to provide accurate and concise updates
        regarding order statuses, service requests, or any other pending items the user might ask about.
        If you require specific information (like an order number, tracking ID, or account details) to provide a status update,
        clearly state what information you need. If the information is not immediately available, explain why and
        suggest alternative ways for the user to get their status (e.g., check their account dashboard or contact a specific department).
        Given the following messages, respond appropriately to the user's request for status, prioritizing clarity and directness.
        """
        llm_messages = create_llm_msg(STATUS_PROMPT, state.messages)
        return {"response": self.model.stream(llm_messages), "category": "status_agent"}

    def feedback_agent(self, state: AgentState):
        print("Feedback agent processing....")
        FEEDBACK_PROMPT = f"""
        You are an attentive feedback collection agent. Your main objective is to solicit and record user feedback
        in a structured and encouraging manner. Thank the user for their input, whether positive or negative,
        and assure them that their feedback is valuable and will be considered for product/service improvement.
        If appropriate, ask follow-up questions to get more detailed insights into their experience.
        Given the following messages, respond appropriately to the user's feedback, showing appreciation for their input.
        """
        llm_messages = create_llm_msg(FEEDBACK_PROMPT, state.messages)
        return {"response": self.model.stream(llm_messages), "category": "feedback_agent"}

# Single Message - Graph

In [15]:
import uuid

In [16]:
# Single message
msg_list=[]

user_input="what is the status of my order?."
msg_list.append({"role":"user","content":user_input})
app=ChatbotAgent(OPENAI_API_KEY)
thread_id = uuid.uuid4()
thread={"configurable":{"thread_id":thread_id}}
full_resp = ""
for s in app.graph.stream({'messages': msg_list}, thread):
  for k,v in s.items():
    if resp_gen := v.get("response"):
      print(f"Assistant: ")
      for chunk in resp_gen:
        text = getattr(chunk, "content", None) or getattr(chunk, "delta", None) # or str(chunk)
        if text:
          print(text, end="", flush=True)
          full_resp += text

if full_resp:
  msg_list.append({"role":"assistant","content":full_resp})

Classified category: status_agent
Status agent processing....
Assistant: 
I can check that. I need your order number to pull up the latest status.

If you don’t have the number handy, you can also share:
- the email used for the order, and
- the last name on the order, plus
- an approximate order date or the shipping address.

You can also view status in your account: Sign in > Orders. If it’s shipped, I can help with the tracking once you provide the tracking number or carrier. 

Please share the order number (or provide the alternative details).

# Chat - Graph

In [17]:
msg_list=[]
while True:
  user_input=input("User: ")
  if(len(user_input)<=0):
    print("\n\n\nGoodbye!!\n\n\n")
    break
  msg_list.append({"role":"user","content":user_input})
  app=ChatbotAgent(OPENAI_API_KEY)
  thread_id = uuid.uuid4()
  thread={"configurable":{"thread_id":thread_id}}
  full_resp = ""
  for s in app.graph.stream({'messages': msg_list}, thread):
    for k,v in s.items():
      if resp_gen := v.get("response"):
        print(f"Assistant: ")
        for chunk in resp_gen:
          text = getattr(chunk, "content", None) or getattr(chunk, "delta", None) # or str(chunk)
          if text:
            print(text, end="", flush=True)
            full_resp += text

  if full_resp:
    msg_list.append({"role":"assistant","content":full_resp})
    print("\n**********\n")

User: Helloi
Classified category: smalltalk_agent
Smalltalk agent processing....
Assistant: 
Hey there! Hello to you too. How’s your day going so far?  
Got any plans or topics you want to chat about—movies, music, a cool fact, or maybe I can crack a quick joke to brighten things up. What’s something you’re into lately?
**********

User: Where is my Order?
Classified category: status_agent
Status agent processing....
Assistant: 
I can help, but I need a bit of detail to look up your order.

Please provide:
- Order number (or tracking ID). If you have neither:
- The email used to place the order
- Your full name on the order
- Approximate order date (and maybe the shipping country)

If you share the order number, I’ll give you the latest status (e.g., Processing, Shipped, In Transit, Out for Delivery, Delivered) and any available tracking info. 

If you don’t have the details, you can also check your orders in your account dashboard under Orders > Track.
**********

User: 



Goodbye!!
